# Notebook 1: Eksploracyjna Analiza Danych (EDA) – PTB-XL

## Cel
Zapoznanie się ze strukturą zbioru danych PTB-XL:
- analiza demograficzna pacjentów
- rozkład klas diagnostycznych (5 superklasowych kategorii)
- wizualizacja przykładowych sygnałów EKG (wszystkie 12 odprowadzeń)
- analiza długości sygnałów i brakujących danych

**Wymaganie:** Kontrola pośrednia nr 1


In [ ]:
# ── Instalacja zależności (jeśli potrzebna) ──────────────────────────────────
# Odkomentuj i uruchom raz:
# !pip install -r ../requirements.txt


In [ ]:
# ── Importy ──────────────────────────────────────────────────────────────────
import os
import sys
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import wfdb

# Dodaj katalog nadrzędny do ścieżki, żeby importować moduły z src/
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ''))
sys.path.insert(0, '..')
from src.data_loader import (
    load_ptbxl_metadata,
    load_scp_statements,
    build_labels,
    load_raw_data,
    SUPERCLASSES,
)

# Konfiguracja wizualizacji
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid')

# ── Ścieżka do danych ─────────────────────────────────────────────────────────
# Zmień tę ścieżkę na lokalizację pobranego zbioru PTB-XL
DATA_PATH = '../data/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3'

print(f"Ścieżka danych: {DATA_PATH}")
print(f"Czy katalog istnieje: {os.path.isdir(DATA_PATH)}")


## 1. Wczytanie metadanych

Plik `ptbxl_database.csv` zawiera informacje o każdym zapisie EKG:
- dane demograficzne pacjenta (wiek, płeć, wzrost, waga)
- kody diagnostyczne SCP w formacie słownika Python
- numer foldu walidacyjnego (`strat_fold` 1–10)
- ścieżki do plików sygnałów (100 Hz i 500 Hz)


In [ ]:
# Wczytaj metadane
Y = load_ptbxl_metadata(DATA_PATH)
print(f"Liczba rekordów EKG: {len(Y):,}")
print(f"\nKolumny DataFrame:\n{list(Y.columns)}")
print(f"\nPierwsze 3 rekordy:")
Y.head(3)


In [ ]:
# Wczytaj i wyświetl kody SCP
agg_df = load_scp_statements(DATA_PATH)
print(f"Liczba kodów SCP: {len(agg_df)}")
print(f"\nKolumny SCP statements:\n{list(agg_df.columns)}")

# Pokaż superklasy diagnostyczne
diag_df = agg_df[agg_df.diagnostic == 1]
print(f"\nKody diagnostyczne: {len(diag_df)}")
print("\nRozkład po superklasach:")
print(diag_df['diagnostic_class'].value_counts())


## 2. Przygotowanie etykiet

Agregujemy szczegółowe kody SCP do 5 superklasowych kategorii diagnostycznych:
- **NORM** – prawidłowy zapis EKG
- **MI**   – zawał mięśnia sercowego (Myocardial Infarction)
- **STTC** – zmiany odcinka ST i fali T (ST/T Change)
- **CD**   – zaburzenia przewodzenia (Conduction Disturbance)
- **HYP**  – przerost serca (Hypertrophy)


In [ ]:
# Buduj etykiety diagnostyczne
Y_labeled = build_labels(Y, DATA_PATH)
print(f"Rekordy z etykietami: {len(Y_labeled):,}")
print(f"\nRozkład etykiet (single-label):")
label_counts = Y_labeled['label_single'].value_counts()
print(label_counts)


In [ ]:
# Wizualizacja rozkładu klas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Wykres słupkowy
colors = ['#2196F3', '#F44336', '#FF9800', '#4CAF50', '#9C27B0']
axes[0].bar(label_counts.index, label_counts.values, color=colors)
axes[0].set_title('Rozkład klas diagnostycznych (superklasy)', fontweight='bold')
axes[0].set_xlabel('Klasa diagnostyczna')
axes[0].set_ylabel('Liczba rekordów')
for i, (cls, cnt) in enumerate(label_counts.items()):
    axes[0].text(i, cnt + 50, str(cnt), ha='center', va='bottom', fontweight='bold')

# Wykres kołowy
axes[1].pie(
    label_counts.values, labels=label_counts.index,
    autopct='%1.1f%%', colors=colors, startangle=90
)
axes[1].set_title('Procentowy udział klas', fontweight='bold')

plt.tight_layout()
plt.savefig('../results/eda_class_distribution.png', bbox_inches='tight', dpi=150)
plt.show()
print(f"\nNiezbalansowanie klas: max/min = {label_counts.max()/label_counts.min():.2f}x")


## 3. Analiza demograficzna pacjentów

In [ ]:
# Podstawowe statystyki demograficzne
print("=== Statystyki demograficzne ===")
print(f"\nWiek (lata):")
print(Y_labeled['age'].describe().round(1))

print(f"\nPłeć (0=kobieta, 1=mężczyzna):")
sex_counts = Y_labeled['sex'].value_counts()
print(sex_counts)
print(f"  Kobiety: {sex_counts.get(0, 0)} ({100*sex_counts.get(0, 0)/len(Y_labeled):.1f}%)")
print(f"  Mężczyźni: {sex_counts.get(1, 0)} ({100*sex_counts.get(1, 0)/len(Y_labeled):.1f}%)")


In [ ]:
# Rozkład wieku i płci
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Histogram wieku
axes[0].hist(Y_labeled['age'].dropna(), bins=30, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(Y_labeled['age'].median(), color='red', linestyle='--', label=f"Mediana: {Y_labeled['age'].median():.0f}")
axes[0].set_title('Rozkład wieku pacjentów', fontweight='bold')
axes[0].set_xlabel('Wiek [lata]')
axes[0].set_ylabel('Liczba rekordów')
axes[0].legend()

# Wiek wg klasy
Y_labeled.boxplot(column='age', by='label_single', ax=axes[1],
                  boxprops=dict(color='steelblue'),
                  medianprops=dict(color='red'))
axes[1].set_title('Rozkład wieku wg klasy diagnostycznej', fontweight='bold')
axes[1].set_xlabel('Klasa diagnostyczna')
axes[1].set_ylabel('Wiek [lata]')

# Płeć wg klasy
sex_class = Y_labeled.groupby(['label_single', 'sex']).size().unstack(fill_value=0)
sex_class.columns = ['Kobiety', 'Mężczyźni']
sex_class.plot(kind='bar', ax=axes[2], color=['#FF69B4', '#4169E1'], alpha=0.85)
axes[2].set_title('Płeć wg klasy diagnostycznej', fontweight='bold')
axes[2].set_xlabel('Klasa')
axes[2].set_ylabel('Liczba pacjentów')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('../results/eda_demographics.png', bbox_inches='tight', dpi=150)
plt.show()


## 4. Wizualizacja sygnałów EKG

Wczytamy kilka przykładowych sygnałów EKG i zwizualizujemy wszystkie 12 odprowadzeń.

**12 odprowadzeń EKG:**
- Kończynowe: I, II, III, aVR, aVL, aVF
- Przedsercowe: V1, V2, V3, V4, V5, V6


In [ ]:
# Wczytaj kilka przykładowych sygnałów EKG (100 Hz)
sample_idx = Y_labeled.groupby('label_single').first().index
sample_df = Y_labeled.loc[sample_idx]
X_sample = load_raw_data(sample_df, sampling_rate=100, data_path=DATA_PATH)

print(f"Kształt danych EKG: {X_sample.shape}")
print(f"  Wymiar 0: liczba rekordów ({X_sample.shape[0]})")
print(f"  Wymiar 1: liczba próbek ({X_sample.shape[1]} = {X_sample.shape[1]/100:.0f}s przy 100 Hz)")
print(f"  Wymiar 2: liczba odprowadzeń ({X_sample.shape[2]})")


In [ ]:
LEAD_NAMES = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
SUPERCLASS_COLORS = {'NORM': '#2196F3', 'MI': '#F44336', 'STTC': '#FF9800', 'CD': '#4CAF50', 'HYP': '#9C27B0'}

def plot_ecg_12lead(signal, fs=100, title='', color='steelblue'):
    """Rysuje pełny 12-odprowadzeniowy zapis EKG w układzie 4x3."""
    fig, axes = plt.subplots(4, 3, figsize=(18, 10), sharex=True)
    t = np.arange(signal.shape[0]) / fs

    for idx, (ax, name) in enumerate(zip(axes.flatten(), LEAD_NAMES)):
        ax.plot(t, signal[:, idx], color=color, linewidth=0.8)
        ax.set_title(f'Odprowadzenie {name}', fontsize=10, fontweight='bold')
        ax.set_ylabel('Amplituda [mV]', fontsize=8)
        ax.grid(True, alpha=0.3)
        if idx >= 9:
            ax.set_xlabel('Czas [s]', fontsize=9)

    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    return fig

# Wyświetl po jednym przykładzie z każdej klasy
for i, (label, row) in enumerate(sample_df.iterrows()):
    cls = row['label_single']
    fig = plot_ecg_12lead(
        X_sample[i], fs=100,
        title=f'Klasa: {cls} | Pacjent: {int(row.patient_id)} | '
              f'Wiek: {int(row.age) if not pd.isna(row.age) else "?"} lat',
        color=SUPERCLASS_COLORS.get(cls, 'steelblue')
    )
    plt.savefig(f'../results/eda_ecg_{cls}.png', bbox_inches='tight', dpi=120)
    plt.show()


## 5. Analiza długości sygnałów i brakujących danych

In [ ]:
# Sprawdź brakujące wartości w metadanych
print("=== Analiza brakujących danych ===")
missing = Y_labeled.isnull().sum()
missing_pct = (missing / len(Y_labeled) * 100).round(2)
missing_df = pd.DataFrame({'Brakujące': missing, 'Procent': missing_pct})
print(missing_df[missing_df['Brakujące'] > 0].sort_values('Procent', ascending=False))


In [ ]:
# Analiza podziału na foldy (train/val/test)
print("=== Podział danych (strat_fold) ===")
fold_dist = Y_labeled['strat_fold'].value_counts().sort_index()
print(fold_dist)

train_size = len(Y_labeled[Y_labeled.strat_fold <= 8])
val_size   = len(Y_labeled[Y_labeled.strat_fold == 9])
test_size  = len(Y_labeled[Y_labeled.strat_fold == 10])
total = len(Y_labeled)

print(f"\nTrening (fold 1-8): {train_size:5,} rekordów ({100*train_size/total:.1f}%)")
print(f"Walidacja (fold 9): {val_size:5,} rekordów ({100*val_size/total:.1f}%)")
print(f"Test (fold 10):     {test_size:5,} rekordów ({100*test_size/total:.1f}%)")


In [ ]:
# Rozkład klas w każdym zbiorze (sprawdź stratyfikację)
splits = {
    'Trening': Y_labeled[Y_labeled.strat_fold <= 8],
    'Walidacja': Y_labeled[Y_labeled.strat_fold == 9],
    'Test': Y_labeled[Y_labeled.strat_fold == 10],
}

print("=== Rozkład klas wg podziału ===")
for split_name, split_df in splits.items():
    counts = split_df['label_single'].value_counts()
    pcts = (counts / len(split_df) * 100).round(1)
    print(f"\n{split_name}:")
    for cls in SUPERCLASSES:
        c = counts.get(cls, 0)
        p = pcts.get(cls, 0)
        print(f"  {cls:5s}: {c:5,} ({p:.1f}%)")


In [ ]:
print("\n=== PODSUMOWANIE EDA ===")
print(f"Całkowita liczba rekordów EKG: {len(Y_labeled):,}")
print(f"Liczba unikalnych pacjentów:   {Y_labeled['patient_id'].nunique():,}")
print(f"Częstotliwość próbkowania:     100 Hz (lr) / 500 Hz (hr)")
print(f"Długość sygnału:               10 sekund (1000 próbek @ 100 Hz)")
print(f"Liczba odprowadzeń:            12")
print(f"Klasy diagnostyczne:           {', '.join(SUPERCLASSES)}")
print(f"Niezbalansowanie klas:         {label_counts.max()/label_counts.min():.1f}x")
